# TripSync AI — 02: Data Cleaning & Preprocessing Pipeline
**Objective:** Transform raw Indian tourism records into clean, leakage-free feature representations with deduplication, macro-thematic clustering, log skewness corrections, and train/val/test splits.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
df_clean = pd.read_csv(PROCESSED_DIR / "places_cleaned.csv")

print(f"Cleaned Places Shape: {df_clean.shape[0]} attractions x {df_clean.shape[1]} features")
df_clean[['name', 'city', 'state', 'macro_category', 'budget_tier', 'rating']].head(5)

## 1. Deduplication and Typo Resolutions

In [ ]:
# Verify zero duplicates remaining
dups = df_clean.duplicated(subset=['name', 'city']).sum()
print(f"Remaining duplicate (name, city) pairs: {dups}")
assert dups == 0, "Duplicate entries remain!"
print("Deduplication verified: exactly 323 unique attractions.")

## 2. 5-Dimensional Thematic Experience Clustering

In [ ]:
print(df_clean['macro_category'].value_counts())
theme_cols = [c for c in df_clean.columns if c.startswith('theme_')]
df_clean[theme_cols].sum()

## 3. Log Skewness Correction (Fees and Popularity)

In [ ]:
print("Raw Entrance Fee Skewness: ", df_clean['entrance_fee_inr'].skew())
print("Log Entrance Fee Skewness: ", df_clean['log_entrance_fee'].skew())
print("Raw Reviews Skewness:      ", df_clean['reviews_lakhs'].skew())
print("Log Reviews Skewness:      ", df_clean['log_reviews'].skew())

## 4. Verification of Leakage-Free Parquet Splits

In [ ]:
train_df = pd.read_parquet(PROCESSED_DIR / "train.parquet")
val_df = pd.read_parquet(PROCESSED_DIR / "val.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "test.parquet")

print(f"Train shape: {train_df.shape}")
print(f"Val shape:   {val_df.shape}")
print(f"Test shape:  {test_df.shape}")

# Check overlap of place_id across splits
train_ids = set(train_df['place_id'])
val_ids = set(val_df['place_id'])
test_ids = set(test_df['place_id'])

assert len(train_ids.intersection(val_ids)) == 0, "Data leakage between Train and Val!"
assert len(train_ids.intersection(test_ids)) == 0, "Data leakage between Train and Test!"
assert len(val_ids.intersection(test_ids)) == 0, "Data leakage between Val and Test!"
print("Strict separation confirmed: ZERO data leakage across splits!")